In [26]:
import torch
import neml2
from pyzag import nonlinear, reparametrization, chunktime
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import os
import re
import tqdm
import pandas as pd
from scipy.optimize import brentq
from neml2.cli.inspect import main as inspect_main
import math

In [27]:
torch.manual_seed(0)

torch.set_default_dtype(torch.double)
if torch.cuda.is_available():
    dev = "cuda:0"
    print("CUDA is available")
    print(f"CUDA version: {torch.version.cuda}")
else:
    dev = "cpu"
device = torch.device(dev)

In [28]:
class SolveStrain(torch.nn.Module):
    """Just integrate the model through some strain history

    Args:
        discrete_equations: the pyzag wrapped model
        nchunk (int): number of vectorized time steps
        rtol (float): relative tolerance to use for Newton's method during time integration
        atol (float): absolute tolerance to use for Newton's method during time integration
    """

    def __init__(self, discrete_equations, nchunk=1, rtol=1.0e-6, atol=1.0e-4, initial_rho_m=4.51):
        super().__init__()
        self.discrete_equations = discrete_equations
        self.nchunk = nchunk
        self.cached_solution = None
        self.rtol = rtol
        self.atol = atol
        self.initial_rho_m = initial_rho_m

    def forward(self, time, temperature, loading, cache=False):
        """Integrate through some time/temperature/strain history and return stress
        Args:
            time (torch.tensor): batched times
            temperature (torch.tensor): batched temperatures
            loading (torch.tensor): loading conditions, which are the input strain in the first base index and then the stress (zero) in the remainder

        Keyword Args:
            cache (bool): if true, cache the solution and use it as a predictor for the next call.
                This heuristic can speed things up during inference where the model is called repeatedly with similar parameter values.
        """
        if cache and self.cached_solution is not None:
            solver = nonlinear.RecursiveNonlinearEquationSolver(
                self.discrete_equations,
                step_generator=nonlinear.StepGenerator(self.nchunk),
                predictor=nonlinear.FullTrajectoryPredictor(self.cached_solution),  # type: ignore
                nonlinear_solver=chunktime.ChunkNewtonRaphson(
                    rtol=self.rtol, atol=self.atol, throw_on_fail=True
                ),
            )
        else:
            solver = nonlinear.RecursiveNonlinearEquationSolver(
                self.discrete_equations,
                step_generator=nonlinear.StepGenerator(self.nchunk),
                predictor=nonlinear.PreviousStepsPredictor(),  # type: ignore
                nonlinear_solver=chunktime.ChunkNewtonRaphson(
                    rtol=self.rtol, atol=self.atol, throw_on_fail=True
                ),
            )

        control = torch.zeros_like(loading)
        control[..., 1:] = 1.0

        # Pack the per-force raw tensors into the single flat forces tensor
        # pyzag consumes. ``assemble_forces`` uses the adapter's own force
        # layout, so the packing is guaranteed to match how the factory later
        # splits the tensor back apart (``_split_by_layout``); base sizes come
        # from the layout, so scalars need no trailing-1 unsqueeze here.
        forces = self.discrete_equations.assemble_forces(
            {
                "t": time,
                "temperature": temperature,
                "fixed_values": loading,
                "control": control,
            }
        )
        state0 = self.discrete_equations.assemble_state(
            {
                "mixed_state": torch.zeros(forces.shape[1:-1] + (6,), device=forces.device),
                "log_rho_m": torch.full(
                    forces.shape[1:-1], math.log(self.initial_rho_m), device=forces.device
                ),
                "back_stress": torch.zeros(forces.shape[1:-1] + (6,), device=forces.device),
            }
        )

        result = nonlinear.solve_adjoint(solver, state0, len(forces), forces)

        if cache:
            self.cached_solution = result.detach().clone()

        if not torch.isfinite(result).all():
            raise RuntimeError(
                "Non-finite state in the integrated solution: the solve produced NaN/Inf "
                "that pyzag's convergence test did not catch. Reduce nchunk or the step size."
            )

        return result[..., 0:1]

    def density(self, result):
        """Physical mobile dislocation density (um^{-2}) from a solved trajectory."""
        return torch.exp(result[..., 6:7])

In [30]:
nmodel = neml2.load_nonlinear_system("tungsten.i", "eq_sys")
nmodel.to(device=device)
pmodel = neml2.pyzag.NEML2PyzagModel(
    nmodel,
    include_parameters=[
        "v_disl_B_k",
        "v_disl_T_0",
        "v_disl_p",
        "v_disl_q",
        "v_disl_H_0",
        "rho_m_rate_k1",
        "rho_m_rate_k2_0",
        "rho_m_rate_Q_d",
        "kinharden_C",
        "kinharden_g",
    ],
)
neml2.compile(pmodel)

model = SolveStrain(pmodel, nchunk=10)
_ = inspect_main(["tungsten.i", "implicit_rate"])

Model: ComposedModel

Inputs (13):
  log_rho_m: type=Scalar
  control: type=SR2
  fixed_values: type=SR2
  mixed_state: type=SR2
  control~1: type=SR2
  fixed_values~1: type=SR2
  mixed_state~1: type=SR2
  temperature: type=Scalar
  t: type=Scalar
  t~1: type=Scalar
  back_stress: type=SR2
  back_stress~1: type=SR2
  log_rho_m~1: type=Scalar

Outputs (3):
  back_stress_residual: type=SR2
  log_rho_m_residual: type=Scalar
  stress_residual: type=SR2

Parameters (50):
  rho_m_from_log.scaling: dtype=torch.float64, device=cpu, shape=[]
  rho_m_from_log.rate: dtype=torch.float64, device=cpu, shape=[]
  E.a: dtype=torch.float64, device=cpu, shape=[]
  E.b: dtype=torch.float64, device=cpu, shape=[]
  E.c: dtype=torch.float64, device=cpu, shape=[]
  nu.a: dtype=torch.float64, device=cpu, shape=[]
  nu.b: dtype=torch.float64, device=cpu, shape=[]
  nu.c: dtype=torch.float64, device=cpu, shape=[]
  L.c_lath: dtype=torch.float64, device=cpu, shape=[]
  L.d_lath: dtype=torch.float64, device=cpu, 

In [ ]:
# Digitized Fig. 10 of Habainy et al. (2015): forged + polished W at 480 C, MST test.
# Columns: x = engineering strain [%], y = engineering stress [MPa].
path = "cyclic_data"
amp_labels = ["0.42", "0.58", "0.75"]

raw_strain = {}  # full digitized curve, strain as a fraction
raw_stress = {}
loop_strain = {}  # closed hysteresis loop only, starting at the +peak
loop_stress = {}

for label in amp_labels:
    (filename,) = [f for f in os.listdir(path) if f.endswith(".csv") and label in f]
    df = pd.read_csv(os.path.join(path, filename))
    # No re-zeroing: the digitized loops are already symmetric about zero strain, and
    # subtracting the first point (a ~0.005% digitizer offset) would shift the whole loop.
    strain = torch.tensor(df["x"].values, device=device) * 1.0e-2  # % -> fraction
    stress = torch.tensor(df["y"].values, device=device)
    raw_strain[label] = strain
    raw_stress[label] = stress

    # The branch rising from (0, 0) is the monotonic first loading drawn under every loop
    # (it lies on the tensile curve in Fig. 10); it is not part of that level's cyclic
    # history. Keep only the closed loop: +peak -> -peak -> back up.
    i_peak = int(torch.argmax(strain))
    loop_strain[label] = strain[i_peak:]
    loop_stress[label] = stress[i_peak:]
    print(
        f"{label}%: {len(strain)} points, loop uses {len(loop_strain[label])} "
        f"(strain {loop_strain[label].min() * 100:.3f}% to {loop_strain[label].max() * 100:.3f}%)"
    )

plt.figure(figsize=(12, 8))
for j, label in enumerate(amp_labels):
    plt.plot(
        raw_strain[label].cpu().numpy() * 100,
        raw_stress[label].cpu().numpy(),
        ":",
        color=f"C{j}",
        alpha=0.5,
        label=f"±{label}% initial branch (dropped)",
    )
    plt.plot(
        loop_strain[label].cpu().numpy() * 100,
        loop_stress[label].cpu().numpy(),
        "-o",
        ms=3,
        color=f"C{j}",
        label=f"±{label}% closed loop (fitted)",
    )
plt.xlabel("Strain [%]")
plt.ylabel("Stress [MPa]")
plt.title("Experimental Tungsten Dataset (Fig. 10, forged + polished, 480 C)")
plt.grid()
plt.legend()
plt.show()

## Multiple Step Test (MST) loading

Habainy et al. (2015), Sec. 2 and 4.3: a **single** forged + polished specimen, cycled fully reversed
(R = -1) at 480 C with strain amplitudes 0.42% -> 0.58% -> 0.75%. Each step runs until the response
saturates; the specimen failed after 9 cycles at the last level. Fig. 10 shows the stabilized loop
at each level.

So the model is driven through **one continuous history** (`nbatch = 1`), so that `log_rho_m` and
`back_stress` carry over between steps, and each digitized loop is compared with the matching cycle:

* blocks 1 and 2: the last cycle (`n_cycles` must be long enough for the model to saturate; check
  by increasing it and confirming the compared loops do not move);
* block 3: an early cycle, with a lower loss weight, since the specimen was probably cracking near
  cycle 9.

The paper gives the rate as "2 mm s^-1". Over the 20 mm reduced section of the high-temperature
specimen (Fig. 1b) that is about 0.1 1/s; it may also have meant 2e-3 1/s, so check both.

In [ ]:
temperature_K = 480.0 + 273.15
strain_rate = 1.0e-1  # 1/s: 2 mm/s over the 20 mm gauge section (Fig. 1b); also try 2.0e-3

amplitudes = [0.0042, 0.0058, 0.0075]  # strain amplitudes (fraction), one per MST step
n_cycles = [10, 10, 9]  # cycles per step; the paper only fixes the last one (failure at 9)
compare_cycle = [n_cycles[0] - 1, n_cycles[1] - 1, 4]  # 0-based cycle compared in each step
loss_weights = [1.0, 1.0, 0.5]

points_per_cycle = 80  # time steps per cycle; must be a multiple of 4
assert points_per_cycle % 4 == 0

In [ ]:
q = points_per_cycle // 4


def one_cycle(amp):
    """One fully reversed triangle cycle 0 -> +amp -> -amp -> 0 at constant |strain rate|."""
    return torch.cat(
        [
            torch.linspace(0.0, amp, q + 1, device=device)[1:],
            torch.linspace(amp, -amp, 2 * q + 1, device=device)[1:],
            torch.linspace(-amp, 0.0, q + 1, device=device)[1:],
        ]
    )


# Step blocks back to back; each block starts and ends at zero strain, so the history is
# continuous. block_start[j] is the time index of the first step of block j (index 0 is t = 0).
segments = []
block_start = []
n = 1
for amp, nc in zip(amplitudes, n_cycles):
    block_start.append(n)
    segments.append(one_cycle(amp).repeat(nc))
    n += len(segments[-1])

strain_history = torch.cat([torch.zeros(1, device=device)] + segments)
# Constant strain rate: time advances with the strain path length.
time_history = (
    torch.cat([torch.zeros(1, device=device), strain_history.diff().abs()]).cumsum(0) / strain_rate
)

ntime = len(strain_history)
nbatch = 1
time = time_history[:, None].clone()
temperature = torch.full_like(time, temperature_K)
loading = torch.zeros((ntime, nbatch, 6), device=device)
loading[:, 0, 0] = strain_history

plt.figure(figsize=(14, 5))
plt.plot(time_history.cpu().numpy(), strain_history.cpu().numpy() * 100, lw=0.8)
for j, start in enumerate(block_start):
    plt.axvline(time_history[start - 1].item(), color="k", ls=":", lw=0.8)
    first = start + compare_cycle[j] * points_per_cycle
    plt.axvspan(
        time_history[first - 1].item(),
        time_history[first + points_per_cycle - 1].item(),
        color=f"C{j}",
        alpha=0.3,
        label=f"Compared cycle, ±{amp_labels[j]}% (cycle {compare_cycle[j] + 1})",
    )
plt.xlabel("Time (s)")
plt.ylabel("Strain [%]")
plt.title("MST strain history (single specimen)")
plt.legend()
plt.grid()
plt.show()

print(
    f"\n--- Full Input Tensors ---\ntime: {time.shape}\ntemperature: {temperature.shape}\nloading: {loading.shape}"
)

In [ ]:
def loop_fraction(strain, full_loop_path):
    """Distance travelled along the strain path, as a fraction of one full loop."""
    s = torch.cat([strain.new_zeros(1), strain.diff().abs()]).cumsum(0)
    return s / full_loop_path


# For each step, take the model's compared cycle starting at its +peak and match it to the
# digitized loop by the fraction of the loop travelled, not by point index. The fraction is
# normalized by each loop's own peak-to-peak strain, so small differences between nominal
# and digitized amplitudes don't misalign the branches.
compare_idx = []  # time indices into the model history, one tensor per step
targets = []  # experimental stress at those indices
target_strain = []  # experimental strain at those indices (for plotting only)
for j, label in enumerate(amp_labels):
    i_peak = block_start[j] + compare_cycle[j] * points_per_cycle + q - 1
    window = torch.arange(i_peak, min(i_peak + points_per_cycle + 1, ntime), device=device)
    u_model = loop_fraction(strain_history[window], 4.0 * amplitudes[j])

    exp_eps = loop_strain[label]
    u_exp = loop_fraction(exp_eps, 2.0 * (exp_eps.max() - exp_eps.min()))

    keep = u_model <= u_exp[-1]  # the digitized loop stops short of closing
    compare_idx.append(window[keep])
    u = u_model[keep].cpu().numpy()
    targets.append(
        torch.tensor(
            np.interp(u, u_exp.cpu().numpy(), loop_stress[label].cpu().numpy()), device=device
        )
    )
    target_strain.append(
        torch.tensor(np.interp(u, u_exp.cpu().numpy(), exp_eps.cpu().numpy()), device=device)
    )
    print(
        f"±{label}%: {len(compare_idx[j])} compared points (time steps {i_peak}..{int(window[keep][-1])})"
    )

# Check: the targets sit on the digitized loop, and model strain matches the data strain.
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
for j, label in enumerate(amp_labels):
    ax[0].plot(
        loop_strain[label].cpu().numpy() * 100,
        loop_stress[label].cpu().numpy(),
        "-",
        color=f"C{j}",
        label=f"Exp ±{label}%",
    )
    ax[0].plot(
        strain_history[compare_idx[j]].cpu().numpy() * 100,
        targets[j].cpu().numpy(),
        "o",
        ms=3,
        color=f"C{j}",
        label=f"Targets ±{label}%",
    )
    ax[1].plot(
        target_strain[j].cpu().numpy() * 100,
        strain_history[compare_idx[j]].cpu().numpy() * 100,
        "o",
        ms=3,
        color=f"C{j}",
        label=f"±{label}%",
    )
ax[0].set_xlabel("Strain [%]")
ax[0].set_ylabel("Stress [MPa]")
ax[0].set_title("Loss targets on the model's time steps")
ax[0].grid()
ax[0].legend()
lim = [-0.8, 0.8]
ax[1].plot(lim, lim, "k--", lw=0.8)
ax[1].set_xlabel("Experimental strain at matched point [%]")
ax[1].set_ylabel("Model strain [%]")
ax[1].set_title("Alignment check (should lie on the diagonal)")
ax[1].grid()
ax[1].legend()
plt.tight_layout()
plt.show()

In [35]:
initial_params = {}
print("--- Before Reparameterization ---")
with torch.no_grad():
    for n, p in model.named_parameters():
        if p.dim() == 0:
            p.data = p.data.reshape(1)
        initial_params[n] = p.data.detach().clone()
        print(f"{n}: {p.data}, {p.shape}\n")

--- Before Reparameterization ---
discrete_equations.v_disl_B_k: tensor([8.3000e-11]), torch.Size([1])

discrete_equations.v_disl_T_0: tensor([3325.5000]), torch.Size([1])

discrete_equations.v_disl_p: tensor([0.6000]), torch.Size([1])

discrete_equations.v_disl_q: tensor([1.4000]), torch.Size([1])

discrete_equations.v_disl_H_0: tensor([2.5500]), torch.Size([1])

discrete_equations.kinharden_C: tensor([19000.]), torch.Size([1])

discrete_equations.kinharden_g: tensor([125.]), torch.Size([1])

discrete_equations.rho_m_rate_k1: tensor([5250.]), torch.Size([1])

discrete_equations.rho_m_rate_k2_0: tensor([525.]), torch.Size([1])

discrete_equations.rho_m_rate_Q_d: tensor([0.0110]), torch.Size([1])



In [ ]:
with torch.no_grad():
    stress = model(time, temperature, loading)
print(stress.shape)

In [ ]:
def plot_against_data(stress, title):
    stress = stress.detach().cpu()
    eps = strain_history.cpu().numpy() * 100
    fig, ax = plt.subplots(1, 2, figsize=(16, 7))

    ax[0].plot(eps, stress[:, 0, 0].numpy(), color="0.8", lw=0.8, label="Model, full history")
    for j, label in enumerate(amp_labels):
        idx = compare_idx[j].cpu()
        ax[0].plot(
            eps[idx],
            stress[idx, 0, 0].numpy(),
            color=f"C{j}",
            lw=2.5,
            label=f"Model ±{label}%, cycle {compare_cycle[j] + 1}",
        )
        ax[0].plot(
            loop_strain[label].cpu().numpy() * 100,
            loop_stress[label].cpu().numpy(),
            "--o",
            ms=3,
            color=f"C{j}",
            label=f"Exp ±{label}%",
        )
    ax[0].set_xlabel("Strain [%]")
    ax[0].set_ylabel("Stress [MPa]")
    ax[0].set_title(title)
    ax[0].grid()
    ax[0].legend(ncol=2, fontsize=8)

    t = time_history.cpu().numpy()
    ax[1].plot(t, stress[:, 0, 0].numpy(), color="0.6", lw=0.8)
    for j, label in enumerate(amp_labels):
        idx = compare_idx[j].cpu()
        ax[1].plot(t[idx], stress[idx, 0, 0].numpy(), color=f"C{j}", lw=2.5)
        ax[1].plot(t[idx], targets[j].cpu().numpy(), "--", color=f"C{j}")
    ax[1].set_xlabel("Time (s)")
    ax[1].set_ylabel("Stress [MPa]")
    ax[1].set_title("Stress vs. time (compared cycles: solid = model, dashed = exp)")
    ax[1].grid()
    plt.tight_layout()
    plt.show()


plot_against_data(stress, "Untrained model vs Fig. 10")

In [12]:
class LogRangeRescale(reparametrization.RangeRescale):
    """Map [0, 1] to [lb, ub] in log space, so an Adam step is a relative change."""

    def __init__(
        self,
        lb: torch.Tensor | float,
        ub: torch.Tensor | float,
        clamp: bool = True,
    ) -> None:
        super().__init__(lb, ub, clamp)
        self.lb = math.log(lb)
        self.ub = math.log(ub)
        self.clamp = clamp

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        if self.clamp:
            X = torch.clamp(X, 0, 1)

        return torch.exp(X * (self.ub - self.lb) + self.lb)

    def reverse(self, X: torch.Tensor) -> torch.Tensor:
        Y = (torch.log(X) - self.lb) / (self.ub - self.lb)

        if self.clamp:
            return torch.clamp(Y, 0, 1)

        return Y

In [13]:
Bk_scaler = LogRangeRescale(
    torch.tensor([1.0e-12], device=device), torch.tensor([1.0e-5], device=device)
)  # MPa*s
T_0_scaler = reparametrization.RangeRescale(
    torch.tensor([100.0], device=device), torch.tensor([5000.0], device=device)
)  # K
p_scaler = LogRangeRescale(torch.tensor([0.01], device=device), torch.tensor([10.0], device=device))
q_scaler = LogRangeRescale(torch.tensor([0.01], device=device), torch.tensor([10.0], device=device))
H0_scaler = reparametrization.RangeRescale(
    torch.tensor([0.1], device=device), torch.tensor([10.0], device=device)
)  # eV
C_scaler = LogRangeRescale(
    torch.tensor([100.0], device=device), torch.tensor([1.0e6], device=device)
)  # MPa
g_scaler = LogRangeRescale(torch.tensor([0.1], device=device), torch.tensor([1.0e4], device=device))
k1_scaler = LogRangeRescale(
    torch.tensor([100.0], device=device), torch.tensor([1.0e7], device=device)
)  # um^-1
k2_0_scaler = LogRangeRescale(
    torch.tensor([0.0001], device=device), torch.tensor([1.0e6], device=device)
)
Q_d_scaler = LogRangeRescale(
    torch.tensor([0.0001], device=device), torch.tensor([1.0], device=device)
)  # eV

model_reparameterizer = reparametrization.Reparameterizer(
    {
        "discrete_equations.v_disl_B_k": Bk_scaler,
        "discrete_equations.v_disl_T_0": T_0_scaler,
        "discrete_equations.v_disl_p": p_scaler,
        "discrete_equations.v_disl_q": q_scaler,
        "discrete_equations.v_disl_H_0": H0_scaler,
        "discrete_equations.kinharden_C": C_scaler,
        "discrete_equations.kinharden_g": g_scaler,
        "discrete_equations.rho_m_rate_k1": k1_scaler,
        "discrete_equations.rho_m_rate_k2_0": k2_0_scaler,
        "discrete_equations.rho_m_rate_Q_d": Q_d_scaler,
    },
    error_not_provided=True,
)
model_reparameterizer(model)
print(f"--- After Reparameterization ---")
initial_params_reparam = {}
for n, p in model.named_parameters():
    initial_params_reparam[n] = p.data.detach().clone()
    print(f"{n}: {p.data}, requires_grad={p.requires_grad}")

--- After Reparameterization ---
discrete_equations.parametrizations.v_disl_B_k.original: tensor([0.2742]), requires_grad=True
discrete_equations.parametrizations.v_disl_T_0.original: tensor([0.6583]), requires_grad=True
discrete_equations.parametrizations.v_disl_p.original: tensor([0.5927]), requires_grad=True
discrete_equations.parametrizations.v_disl_q.original: tensor([0.7154]), requires_grad=True
discrete_equations.parametrizations.v_disl_H_0.original: tensor([0.2475]), requires_grad=True
discrete_equations.parametrizations.kinharden_C.original: tensor([0.5697]), requires_grad=True
discrete_equations.parametrizations.kinharden_g.original: tensor([0.6194]), requires_grad=True
discrete_equations.parametrizations.rho_m_rate_k1.original: tensor([0.3440]), requires_grad=True
discrete_equations.parametrizations.rho_m_rate_k2_0.original: tensor([0.6720]), requires_grad=True
discrete_equations.parametrizations.rho_m_rate_Q_d.original: tensor([0.5103]), requires_grad=True


In [ ]:
niter = 1500

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer=optimizer,
    mode="min",
    factor=0.1,
    patience=150,
    threshold=1e-4,
)


def cyclic_loss(res):
    """Weighted MSE over the compared cycles, each normalized by its own peak stress."""
    total = 0.0
    for j in range(len(amp_labels)):
        pred = res[compare_idx[j], 0, 0]
        scale = targets[j].abs().max()
        total = total + loss_weights[j] * torch.mean(((pred - targets[j]) / scale) ** 2)
    return total / sum(loss_weights)


titer = tqdm.tqdm(
    range(niter),
    bar_format="{desc}: {percentage:3.0f}%|{bar}|{n_fmt}/{total_fmt}{postfix}",
)
titer.set_description("Loss:")
loss_history = []

for i in titer:
    optimizer.zero_grad()
    res = model(time, temperature, loading)
    loss = cyclic_loss(res)

    # --- Bug fix: guard BEFORE backward ---
    if not torch.isfinite(loss) or loss.item() > 1e6:
        titer.set_description(f"Loss: [SKIP bad solve @ iter {i}]")
        continue  # skip backward + step entirely

    loss.backward()
    optimizer.step()
    scheduler.step(loss.item())

    loss_history.append(loss.detach().clone().cpu())
    titer.set_description("Loss: %3.2e" % loss_history[-1])

# --- Plots ---
plt.figure()
plt.loglog(loss_history, label="Training")
plt.xlabel("Iteration")
plt.ylabel("MSE")
plt.legend()
plt.title("Loss history")

plt.figure()
plt.plot(loss_history)
plt.xlabel("Iteration")
plt.ylabel("MSE")
plt.title("Loss history (linear scale)")
plt.show()

In [ ]:
plot_against_data(res, "Trained model vs Fig. 10")